# Part 7c: Validation
We executed NN inference on the pynq-z2! Now we can copy the `y_hw.npy` back to the host we've been using for the training and synthesis, and make a final plot to check that the output we took on the board is as expected.

The command to copy it back is

```bash
scp xilinx@192.168.2.99:~/jupyter_notebooks/y_hw.npy model_3/
```

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import plotting

%matplotlib inline
from sklearn.metrics import accuracy_score

y_hw = np.load('model_3/y_hw.npy')
y_test = np.load('y_test.npy')
classes = np.load('classes.npy', allow_pickle=True)
y_hls = np.load('model_3/y_hls.npy')
y_qkeras = np.load('model_3/y_qkeras.npy')

print("Accuracy QKeras, CPU:     {}".format(accuracy_score(np.argmax(y_test, axis=1), np.argmax(y_qkeras, axis=1))))
print("Accuracy hls4ml, CPU: {}".format(accuracy_score(np.argmax(y_test, axis=1), np.argmax(y_hls, axis=1))))
print("Accuracy hls4ml, pynq-z2: {}".format(accuracy_score(np.argmax(y_test, axis=1), np.argmax(y_hw, axis=1))))

fig, ax = plt.subplots(figsize=(9, 9))
_ = plotting.makeRoc(y_test, y_qkeras, classes, linestyle='-')
plt.gca().set_prop_cycle(None)  # reset the colors
_ = plotting.makeRoc(y_test, y_hls, classes, linestyle='--')
plt.gca().set_prop_cycle(None)  # reset the colors
_ = plotting.makeRoc(y_test, y_hw, classes, linestyle='-.')

from matplotlib.lines import Line2D

lines = [Line2D([0], [0], ls='-'), Line2D([0], [0], ls='--'), Line2D([0], [0], ls='-.')]
from matplotlib.legend import Legend

leg = Legend(ax, lines, labels=['QKeras, CPU', 'hls4ml, CPU', 'hls4ml, pynq-z2'], loc='lower right', frameon=False)
ax.add_artist(leg)

## 📦 Data preservation & provenance with Dataerai

The cell below preserves **this notebook's** artifacts as versioned Dataerai assets in a **per-notebook collection**, links them into the shared lineage DAG (with verifiable DID citations), and adds a static **recording** — the notebook file plus its execution log — using the existing Dataerai *beta* APIs (no backend changes).

It is **idempotent** and safe to re-run; running the parts in order accumulates the full provenance graph into `PROVENANCE.md` / `provenance_manifest.json`. See **[DATAERAI_PROVENANCE.md](DATAERAI_PROVENANCE.md)** for one-time setup.

In [ ]:
# Dataerai — preserve this notebook's artifacts into a per-notebook collection,
# link the lineage DAG, and record the notebook + execution log.
# No-op unless `dataerai_hls4ml` is importable and DATAERAI_PROVENANCE != 0.
# Setup (see DATAERAI_PROVENANCE.md):
#   dataerai auth login --server https://beta.dataerai.com
#   export DATAERAI_PROJECT_ID=<your-project-uuid>
# Or run fully offline (records lineage locally, uploads nothing): export DATAERAI_DRY_RUN=1
try:
    import dataerai_hls4ml as dp
except ImportError:
    dp = None

if dp and dp.enabled():
    manifest = dp.capture(notebook="part7c_validation")
    mode = 'dry-run' if manifest['dry_run'] else manifest['server']
    print(f"Dataerai provenance: run {manifest['run_id']} — "
          f"{len(manifest['artifacts'])} assets, {len(manifest['edges'])} edges ({mode})")
    print('Wrote PROVENANCE.md and provenance_manifest.json')